# Extracting member names and tasks from raw attributions with fuzzy matching


This notebook was created in the early stages of extracting member names and task descriptions from raw attribution texts. The idea was to separate the raw text into chunks based on member names in the raw text that were fuzzy matched with names from the roster. Since these chunks would not be perfect (due to some incorrectly matched names) they would then be passed to an LLM with some overlap in prompts. In this way, the LLM would process smaller prompts with less irrelevant text, and it would produce fewer hallucinations.

However, this approach only works with certain attribution structures (e.g., when teams write task descriptions after each member's name, or when names are composed of only one first and one last name). Because each team structures its attributions differently, we ultimately did not use the chunking strategy presented here.

Instead, a large-context LLM was used in `9_llm_extract_names_and_task_descriptions.ipynb` to process full raw attribution texts without chunking. This `7_fuzzy_extract_names_and_tasks.ipynb` notebook is retained because some of the fuzzy matching logic may be useful in later stages of the project.

In [2]:
import requests
import re
import difflib
import pandas as pd
from pathlib import Path
import json
from tqdm import tqdm
import subprocess
from rapidfuzz import process, fuzz

In [ ]:
roster_input_path = "../data/attributions/2022_attributions/team_rosters_2022.json"

with open(roster_input_path, "r", encoding="utf-8") as f:
    roster_2022_dict = json.load(f)

In [7]:
roster_2022_dict

{'AFCM-Egypt': ['Moetaz Sherif Mohamed Radwan Metawea',
  'Omar Ahmed Abdalla',
  'Ahmad Mahmoud Galal',
  'Ahmed Elshazly',
  'Mahmoud Mohammed AbdelGawad',
  'Mohamed Osama Mohamed Moawad',
  'Mohammad Tarek Mansour',
  'Tamer Ashry',
  'Yasser Elbedewy',
  'Khaled Shokry',
  'Ayman Shawky',
  'Ahmed Adel Rezk',
  'Ahmed Eldahshan',
  'Ahmed Gamal Mohamed Mattar',
  'Ahmed Shalan',
  'Ahmed Wael',
  'Ahmed Wael Mansour',
  'Hossam Algamal',
  'Hossam Eldeen Bannis',
  'Khalid Hela',
  'Mahmoud Mohamed Abd-Elmonem',
  'mahmoud sayed',
  'Mohamed Aboelghar',
  'Mohamed Emad Abd El-Wahab',
  'Mohamed Hesham Abdulhai',
  'Mohamed Sayed Hasouna',
  'Omar Emad Hassan Elsabaye'],
 'AHS_Peking': ['Lingfang Tang',
  'Lintong Zhao',
  'Yilan Li',
  'Amelia Siqi Huang',
  'MIngde Xu'],
 'ASIJ_Tokyo': ['Beth Crissy',
  'Ai Okura',
  'Annika Elstrom',
  'Annmarie Hashimoto',
  'Ei Fukumoto',
  'Hana Ito',
  'Julia Shikuma',
  'Kai Hyodo',
  'Kian Benner',
  'Koharu Matsuki',
  'Mia Hamaguchi',
  

## 1. Get names and raw attributions text for a team

In [4]:
# Helper functions

# Load the JSON file containing all teams' attributions data
def load_attributions_json(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

# Return team's full attribution text from raw attributions data
def get_team_attributions_text(data, team_name):
    for entry in data:
        if entry["teamName"] == team_name:
            return entry["attributionsText"]
    return None

# Return the list of member names for given team from the roster dictionary
def get_team_roster(roster_dict, team_name):
    return roster_dict.get(team_name, [])

In [5]:
# Get attributions text and team members for Thessaly for example
filepath = "../data/attributions/2022_attributions/raw_attributions_2022.json"

data = load_attributions_json(filepath)

test_team_name = "Thessaly"

test_attributions_text = get_team_attributions_text(data, test_team_name)

if test_attributions_text is None:
    print(f"No attributions text found for team: {test_team_name}")

test_member_names = get_team_roster(roster_2022_dict, test_team_name)

In [6]:
test_member_names

['Asteria Tsapadikou',
 'Ioanna Gkoni',
 'Konstantinos Elenis',
 'Afrodite Katsaouni',
 'KALLIAMPAKOU AIKATERINI',
 'Kostas Mathiopoulos',
 'Antonis G',
 'Artemis Hatzigeorgiou',
 'Elli Magkouta',
 'Anastasios Platis',
 'Christina Theofanopoulou',
 'Eleftheria Lakaki',
 'Georgios Melachroinos',
 'Ioannis Retalis',
 'Katerina Patrinou',
 'Natalia Papadimitriou',
 'Stefanos Digenis',
 'Theodore Papanikolas']

In [7]:
test_attributions_text

'Home\nProject\nDescription Safety Proof of Concept Implementation Contribution\nWet Lab\nDesign Parts Experiments Engineering Improvement Interlab\nDry Lab\nModeling Hardware Software\nHuman Practices\nIntegrated Human Practices Science Communication Education Sustainable Development\nTeam\nMembers Attributions Collaborations Partnership\nJamboree\nIndex\nAttributions\nOverview\nTeam\nWet Lab\nDry Lab\nHuman Practices\nFundraising\nOverview\nOur project would not have progressed without the help and advice of\n                exceptional people, inside and outside our university. Therefore, it is\n                crucial for us to thank everyone who contributed to our trip. We have\n                been incredibly fortunate to be surrounded by supportive and helpful\n                people, including of course our advisors, instructors, and PI’s.\nTeam\nMembers\nNatalie Papadimitriou:\nBiochemistry and Biotechnology undergraduate\n                      student at University of Thessal

## 2. Split attributions text into chunks by member names

In [6]:
# Text Cleaning functions

# Replace newlines with single spaces so there are no names split by a new line that should be fuzzy matched

def flatten_text(text):
    return re.sub(r'\s+', ' ', text).strip()

# Normalize names in the roster and attributions text (lowercase, strip leading/trailing spaces, remove double blanks)
def normalize_name(name):
    return " ".join(name.lower().strip().split())

In [ ]:
# Fuzzy match names from the roster with the ones from the input flattened text (so attributions text can be split into chunks by name) 

def fuzzy_match_names(text, roster_names, threshold=90):
    """
    Fuzzy match roster names against paragraphs in text, and extract the matched name variants.
    
    - fuzzy  matches full names first
    - fuzzy matches reversed names
    - fuzzy matches first or last names only

    Returns:
        fuzzy_map: roster_name: found_variant_name_in_text
    """
    fuzzy_map = {}

    # Split the text into paragraphs (by newlines)
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]

    for roster_name in roster_names:
        norm_roster = normalize_name(roster_name)
        parts = norm_roster.split() # split into parts (first and last name)

        # Generate different variants of the roster names to match to attributions paragraphs later
        variants = set()
        variants.add(norm_roster) #full name
        if len(parts) == 2:
            variants.add(f"{parts[1]} {parts[0]}") #reversed name
        for part in parts:
            # if len(part) >= 2: # doesn't fix Antonis G. problem
               variants.add(part) #first or last name only

        best_paragraph = None
        best_score = 0

        for para in paragraphs:
            norm_para = normalize_name(para)
            for variant in variants:
                # match variants to normalized paragraphs - find best paragraph to later extract the name from it
                score = fuzz.partial_ratio(variant, norm_para)
                if score > best_score:
                    best_score = score
                    best_paragraph = para

        if best_score >= threshold and best_paragraph:
            extracted_variant = extract_name_variant(best_paragraph, parts)
            # extracted_variant = extract_name_variant(best_paragraph, roster_name) #uncomment for second version
            if extracted_variant:
                fuzzy_map[roster_name] = extracted_variant
            else:
                fuzzy_map[roster_name] = roster_name 

    return fuzzy_map

def extract_name_variant(paragraph, roster_parts):
    """
    Extract a name variant from a paragraph,
    matching words from the roster name.
    """

    # Break paragraph into words
    words = paragraph.split()

    # Look for pairs of title-case words (2 word candidates)
    # issue:  “Instructors Afrodite” would also be a candidate (instead of their full name)
    candidates = []
    for i in range(len(words) - 1):
        w1, w2 = words[i], words[i+1]
        if w1.istitle() and w2.istitle():
            candidate = f"{w1} {w2}"
            candidates.append(candidate)
            candidates.append(f"{w2} {w1}")

    # Single words candidates too
    for w in words:
        if w.istitle():
            candidates.append(w)

    roster_parts_lower = [p.lower() for p in roster_parts]

    for candidate in candidates:
        candidate_norm = normalize_name(candidate)
        for part in roster_parts_lower:
            if part in candidate_norm:
                return candidate.strip()

    return None

In [8]:
# Split text into chunks by the name variants in text

def split_text_by_fuzzy_variants(text, fuzzy_map):
    """
    Splits text into chunks by locating the fuzzy matched name variants in text.
    Each chunk is assigned to the roster name whose variant it follows.
    """

    if not fuzzy_map:
        return []

    matches = []

    for canonical, variant in fuzzy_map.items():

        # Search all occurrences of this variant in text
        for m in re.finditer(re.escape(variant), text, flags=re.IGNORECASE):
            matches.append({
                "start": m.start(),
                "end": m.end(),
                "variant": variant,
                "rosterName": canonical
            })

    if not matches:
        # No matches found
        return [{"rosterName": None, "text": text.strip()}]

    matches = sorted(matches, key=lambda x: x["start"])

    chunks = []
    for i, match in enumerate(matches):
        start_of_chunk = match["end"]
        end_of_chunk = matches[i + 1]["start"] if i + 1 < len(matches) else len(text)

        chunk_text = text[start_of_chunk:end_of_chunk].strip()
        # Include the variant name as the first line of the chunk:
        chunk_text = match["variant"] + "\n" + chunk_text

        chunks.append({
            "rosterName": match["rosterName"],
            "text": chunk_text
        })

    return chunks

In [9]:
# Pipeline

def process_team_attributions(text, roster_names, fuzzy_threshold=80):
    text_flat = flatten_text(text)

    fuzzy_map = fuzzy_match_names(
        text_flat,
        roster_names,
        threshold=fuzzy_threshold
    )

    chunks = split_text_by_fuzzy_variants(text_flat, fuzzy_map)

    matched_names = list(fuzzy_map.keys())
    unmatched_names = [name for name in roster_names if name not in fuzzy_map]

    return chunks, matched_names, unmatched_names, fuzzy_map

In [10]:
# Print chunks function 

def print_chunks(chunks):
    for i, chunk in enumerate(chunks, 1):
        print("=" * 80)
        print(f"Chunk {i}:")
        print("Roster Name:", chunk["rosterName"])
        print(chunk["text"])

In [ ]:
# Execution

chunks, matched, unmatched, fuzzy_map = process_team_attributions(
    test_attributions_text,
    test_member_names,
    fuzzy_threshold=80,
)

print_chunks(chunks)

Chunk 1:
Roster Name: Antonis G
Lab Design
Parts Experiments Engineering Improvement Interlab Dry Lab Modeling Hardware Software Human Practices Integrated Human Practices Science Communication Education Sustainable Development Team Members Attributions Collaborations Partnership Jamboree Index Attributions Overview Team Wet Lab Dry Lab Human Practices Fundraising Overview Our project would not have progressed without the help and advice of exceptional people, inside and outside our university. Therefore, it is crucial for us to thank everyone who contributed to our trip. We have been incredibly fortunate to be surrounded by supportive and helpful people, including of course our advisors, instructors, and PI’s. Team Members
Chunk 2:
Roster Name: Natalia Papadimitriou
Natalie Papadimitriou:
Biochemistry and Biotechnology undergraduate student at University of Thessaly. Head of Graphic Design department and member of the Social Media team. She is the creator and designer of the education

In [19]:
fuzzy_map

{'Asteria Tsapadikou': 'Advisors Asteria',
 'Ioanna Gkoni': 'Ιoanna Gkoni:',
 'Konstantinos Elenis': 'Konstantinos Elenis:',
 'Afrodite Katsaouni': 'Instructors Afrodite',
 'KALLIAMPAKOU AIKATERINI': 'Kalliampakou Aikaterini:',
 'Kostas Mathiopoulos': 'Konstantinos Mathiopoulos:',
 'Antonis G': 'Lab Design',
 'Artemis Hatzigeorgiou': 'Hatzigeorgiou Artemis:',
 'Elli Magkouta': 'Elli Magkouta:',
 'Anastasios Platis': 'Events. Anastasios',
 'Christina Theofanopoulou': 'Fundraising. Christina',
 'Eleftheria Lakaki': 'Eleftheria Lakaki:',
 'Georgios Melachroinos': 'Fundraising. Georgios',
 'Ioannis Retalis': 'Yannos Retalis:',
 'Katerina Patrinou': 'Katerina Patrinou:',
 'Natalia Papadimitriou': 'Natalie Papadimitriou:',
 'Stefanos Digenis': 'Practices. Stefanos',
 'Theodore Papanikolas': 'Theodoros Papanikolas:'}

In [54]:
unmatched

[]